# Energy Optimization — All-Scenarios Driver

Drives the **canonical** pipeline (`Optimizer_MINLP.ipynb` + `post_process_outputs.py`)
once per scenario. No optimizer logic is reimplemented here — scenarios are applied
by patching `cfg["master_pi_data"]` (prices) and `cfg["variables"]` (bound patches)
on a per-run copy of the optimizer notebook.

**Per-scenario outputs** are snapshotted into `scenario_runs/<scenario>/`:
- `optimizer.ipynb` (executed)
- `*_output_v3.xlsx` (optimizer result)
- 7 DB-schema CSVs from `tables_from_db/outputs/`
- step-validation, parity, and QC reports
- `run_metadata.json`

**Combined output:** `eo_all_scenarios_<ts>.xlsx` at root.

**Wall time:** ~30 min × N scenarios.


In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import json, shutil, subprocess, sys, time, traceback
from pathlib import Path
from datetime import datetime
from copy import deepcopy

import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

ROOT = Path('.').resolve()
OPT_NB_SRC      = ROOT / 'Optimizer_MINLP.ipynb'
POST_SCRIPT     = ROOT / 'post_process_outputs.py'
RUNS_DIR        = ROOT / 'scenario_runs'
PARAMS_FILE     = ROOT / '_scenario_params.json'   # written per scenario, read by injected cell
EXECUTE_TIMEOUT = 2700   # seconds per cell (~45 min ceiling for the GEKKO solve cell)

assert OPT_NB_SRC.exists(),  f'Missing {OPT_NB_SRC}'
assert POST_SCRIPT.exists(), f'Missing {POST_SCRIPT}'
RUNS_DIR.mkdir(exist_ok=True)
print(f'ROOT: {ROOT}')
print(f'Driver ready. Optimizer template: {OPT_NB_SRC.name}')


In [ ]:
# ── SCENARIO CONFIGS + RUN LIST ──────────────────────────────────────────────
_BASELINE = {'Power_Rate': 13.334, 'Fuel_Rate': 2.04, 'DMW_Rate': 7.34 / 3.75}
_ALL_TURBINES = [
    'BFW_B_Turb_Status', 'BFW_C_Turb_Status', 'BFW_E_Turb_Status',
    'VHP_BFW_B_Turb_Status', 'VHP_BFW_C_Turb_Status',
    'CW_Turbine_A_Status', 'CW_Turbine_B_Status', 'CW_Turbine_G_Status',
    'Air_Compressor_Turbine_A_Status', 'Air_Compressor_Turbine_D_Status',
    'DMW_Turbine_A_Status', 'DMW_Turbine_C_Status',
]

SCENARIO_CONFIGS = {
    'baseline':    {'prices': dict(_BASELINE), 'bound_patches': [],
                    'description': 'Baseline prices'},
    'power_low':   {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 0.85}, 'bound_patches': []},
    'power_high':  {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 1.15}, 'bound_patches': []},
    'power_vhigh': {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 1.30}, 'bound_patches': []},
    'power_vlow':  {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 0.70}, 'bound_patches': []},
    'fuel_low':    {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 0.85}, 'bound_patches': []},
    'fuel_high':   {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 1.15}, 'bound_patches': []},
    'fuel_vhigh':  {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 1.30}, 'bound_patches': []},
    'fuel_vlow':   {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 0.70}, 'bound_patches': []},
    'dmw_low':     {'prices': {**_BASELINE, 'DMW_Rate':   _BASELINE['DMW_Rate']   * 0.85}, 'bound_patches': []},
    'dmw_high':    {'prices': {**_BASELINE, 'DMW_Rate':   _BASELINE['DMW_Rate']   * 1.15}, 'bound_patches': []},
    'fuel_10x':    {'prices': {**_BASELINE, 'Fuel_Rate': _BASELINE['Fuel_Rate'] * 10.0},
                    'bound_patches': [], 'description': 'Stress: Fuel at 10x baseline'},
    'negative_power': {'prices': {**_BASELINE, 'Power_Rate': -5.0},
                    'bound_patches': [], 'description': 'Stress: grid pays plant'},
    # Infeasibility tests (excluded from default list)
    'infeas_3_boilers': {'prices': dict(_BASELINE),
        'bound_patches': [{'tag_name': f'BLR_{i}_Status', 'lower': 0.0, 'upper': 0.0} for i in (1,2,3)],
        'description': 'Infeasibility: BLR_1/2/3 forced offline'},
    'infeas_all_turbines': {'prices': dict(_BASELINE),
        'bound_patches': [{'tag_name': t, 'lower': 0.0, 'upper': 0.0} for t in _ALL_TURBINES],
        'description': 'Infeasibility: all 12 steam turbines OFF'},
    'infeas_1_boiler': {'prices': dict(_BASELINE),
        'bound_patches': [{'tag_name': 'Total_Boilers_Running', 'lower': 1.0, 'upper': 1.0}],
        'description': 'Infeasibility: pin to single boiler'},
}

SCENARIOS_TO_RUN = [
    'baseline',
    'power_low', 'power_high', 'power_vhigh', 'power_vlow',
    'fuel_low',  'fuel_high',  'fuel_vhigh',  'fuel_vlow',
    'dmw_low',   'dmw_high',
    'fuel_10x',  'negative_power',
]

CONTINUE_ON_ERROR = True
SKIP_POST_PROCESS = False   # set True to run only the optimizer (faster diagnostics)

print(f'Scenarios to run: {len(SCENARIOS_TO_RUN)}')
for s in SCENARIOS_TO_RUN: print(f'  - {s}')


In [ ]:
# ── Per-scenario optimizer notebook builder ──────────────────────────────────
# Loads the canonical Optimizer_MINLP.ipynb, inserts a SCENARIO injection cell
# right after "TASK 1 — DATA INGESTION", and writes a runnable copy.

INJECTION_CELL_SRC = '# ────────────────────────────────────────────────────────────────────────────\n# SCENARIO PARAM INJECTION (driver-inserted, harmless if no JSON present)\n# ────────────────────────────────────────────────────────────────────────────\nimport json as _json, os as _os\n_PARAMS_PATH = _os.path.join(_os.getcwd(), \'_scenario_params.json\')\nif _os.path.exists(_PARAMS_PATH):\n    with open(_PARAMS_PATH, \'r\', encoding=\'utf-8\') as _f:\n        _PARAMS = _json.load(_f)\n    print(f\'  >>> SCENARIO INJECTION ACTIVE: {_PARAMS.get("scenario","?")}\')\n\n    # 1. Prices live in the \'inferred\' sheet as literal-number formulas\n    #    (e.g. Power_Rate.formula_expression = \'13.334000\'). Patch those.\n    _BASELINE_FUEL = 2.04\n    _FUEL_COST_IN_MMBTU_BASELINE = 2.15\n    _prices = _PARAMS.get(\'prices\', {}) or {}\n    if _prices:\n        _df_inf = cfg[\'inferred\']\n        for _tag, _val in _prices.items():\n            _m = _df_inf[\'tag_name\'].astype(str).str.strip() == _tag\n            if not _m.any():\n                print(f\'    skip price (tag missing in inferred): {_tag}\')\n                continue\n            _df_inf.loc[_m, \'formula_expression\'] = f\'{float(_val):.6f}\'\n        if \'Fuel_Rate\' in _prices:\n            _fcm = _prices[\'Fuel_Rate\'] * _FUEL_COST_IN_MMBTU_BASELINE / _BASELINE_FUEL\n            _m = _df_inf[\'tag_name\'].astype(str).str.strip() == \'Fuel_Cost_in_MMBTU\'\n            if _m.any():\n                _df_inf.loc[_m, \'formula_expression\'] = f\'{float(_fcm):.6f}\'\n        cfg[\'inferred\'] = _df_inf\n        print(f\'    prices applied via inferred-formula override: {_prices}\')\n\n    # 2. Patch variables bounds for each bound_patch\n    _patches = _PARAMS.get(\'bound_patches\', []) or []\n    if _patches:\n        _df_v = cfg[\'variables\']\n        _n = 0\n        for _p in _patches:\n            _tag = _p.get(\'tag_name\',\'\')\n            _mask = _df_v[\'tag_name\'].astype(str).str.strip() == _tag\n            if not _mask.any():\n                print(f\'    bound_patch SKIPPED (tag not in variables sheet): {_tag}\')\n                continue\n            if _p.get(\'lower\') is not None:\n                _lo = float(_p[\'lower\'])\n                _df_v.loc[_mask, \'lower_bound_value\']      = _lo\n                _df_v.loc[_mask, \'lower_bound_expression\'] = f\'{_lo}\'\n            if _p.get(\'upper\') is not None:\n                _up = float(_p[\'upper\'])\n                _df_v.loc[_mask, \'upper_bound_value\']      = _up\n                _df_v.loc[_mask, \'upper_bound_expression\'] = f\'{_up}\'\n            _n += 1\n        cfg[\'variables\'] = _df_v\n        print(f\'    bound patches applied: {_n}/{len(_patches)}\')\nelse:\n    print(\'  >>> No _scenario_params.json — running canonical (no scenario overrides)\')\n'

def _find_ingestion_cell_idx(nb):
    """Return the index of the 'TASK 1 - DATA INGESTION' cell in the optimizer."""
    for i, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        src = ''.join(c.source) if isinstance(c.source, list) else c.source
        if 'TASK 1' in src and 'DATA INGESTION' in src:
            return i
    raise RuntimeError("Couldn't find data-ingestion cell in optimizer notebook")

def prepare_optimizer_nb(scenario_dir: Path) -> Path:
    nb = nbformat.read(str(OPT_NB_SRC), as_version=4)
    insert_at = _find_ingestion_cell_idx(nb) + 1
    inj = nbformat.v4.new_code_cell(source=INJECTION_CELL_SRC)
    # Strip any nbformat-required ID metadata that confuses some kernels
    inj.metadata = {}
    nb.cells.insert(insert_at, inj)
    out = scenario_dir / 'optimizer.ipynb'
    nbformat.write(nb, str(out))
    return out

print('Notebook patcher ready.')


In [ ]:
# ── Per-scenario pipeline runner ─────────────────────────────────────────────
# Per-scenario flow:
#   write _scenario_params.json
#   prepare optimizer.ipynb (patched copy)
#   nbconvert execute (writes outputs to ./output/, ./tables_from_db/, root reports)
#   subprocess post_process_outputs.py
#   snapshot all artifacts into scenario_runs/<name>/
#   parse model_output.csv for objective / saving

def _latest(glob_pat):
    paths = sorted(ROOT.glob(glob_pat), key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0] if paths else None

def _snapshot_outputs(scenario_dir: Path, started_at: float):
    """Move per-run artifacts produced after `started_at` into scenario_dir."""
    captured = {}
    # 1. Optimizer output excel  (output/*_output_v3.xlsx)
    out_xlsx = _latest('output/*_output_v3.xlsx')
    if out_xlsx and out_xlsx.stat().st_mtime >= started_at:
        dst = scenario_dir / out_xlsx.name
        shutil.copy2(out_xlsx, dst); captured['optimizer_xlsx'] = str(dst)

    # 2. The 7 DB-schema CSVs
    csv_dir = ROOT / 'tables_from_db' / 'outputs'
    if csv_dir.exists():
        sub = scenario_dir / 'tables_from_db_outputs'
        sub.mkdir(exist_ok=True)
        for csv in csv_dir.glob('*.csv'):
            if csv.stat().st_mtime >= started_at:
                shutil.copy2(csv, sub / csv.name)
        captured['csv_dir'] = str(sub)

    # 3. Validation / parity / QC reports + run_metadata
    for pat in ('step_validation_*.xlsx', 'python_vs_db_parity_*.xlsx',
                'optimizer_qc_report_*.xlsx', 'run_metadata_*.json'):
        p = _latest(pat)
        if p and p.stat().st_mtime >= started_at:
            shutil.copy2(p, scenario_dir / p.name)
            captured.setdefault('reports', []).append(p.name)

    return captured

def _parse_model_output(scenario_dir: Path):
    """Return {'baseline_obj','optimum_obj','saving','saving_pct'} from model_output.csv."""
    import pandas as pd
    csv = scenario_dir / 'tables_from_db_outputs' / 'model_output.csv'
    out = {'baseline_obj': float('nan'), 'optimum_obj': float('nan'),
           'saving': float('nan'), 'saving_pct': float('nan'),
           'objective_tag': None}
    if not csv.exists():
        return out
    try:
        df = pd.read_csv(csv)
        # The objective tag is the one tied to feature_file objective sheet — usually
        # "Objective_2" or similar. We pull both actual+optimum columns for the tag
        # whose absolute actual is largest (proxy for total $/hr objective).
        if {'tag_id','actual','optimum'}.issubset(df.columns):
            # Some objective rows are name-tagged in 'tag_name' if joined; otherwise
            # try a reasonable filter
            if 'tag_name' in df.columns:
                candidates = df[df['tag_name'].astype(str).str.contains('Objective', case=False, na=False)]
                if not candidates.empty:
                    row = candidates.iloc[candidates['actual'].abs().argmax()]
                    out['objective_tag'] = str(row['tag_name'])
                    out['baseline_obj'] = float(row['actual'])
                    out['optimum_obj']  = float(row['optimum'])
            if pd.isna(out['baseline_obj']):
                # Fallback: largest |actual| numeric row
                row = df.iloc[df['actual'].abs().fillna(0).argmax()]
                out['baseline_obj'] = float(row.get('actual', float('nan')))
                out['optimum_obj']  = float(row.get('optimum', float('nan')))
        if not (pd.isna(out['baseline_obj']) or pd.isna(out['optimum_obj'])):
            sv = out['baseline_obj'] - out['optimum_obj']
            out['saving'] = sv
            if abs(out['baseline_obj']) > 1e-6:
                out['saving_pct'] = sv / out['baseline_obj'] * 100
    except Exception as e:
        out['parse_error'] = str(e)
    return out

def run_scenario(name: str) -> dict:
    cfg_s = SCENARIO_CONFIGS[name]
    scenario_dir = RUNS_DIR / name
    scenario_dir.mkdir(exist_ok=True)
    started = time.time()
    print(f'\n{"#"*70}\n# SCENARIO: {name}')
    print(f'#   prices = {cfg_s["prices"]}')
    if cfg_s.get('bound_patches'):
        print(f'#   bound_patches = {len(cfg_s["bound_patches"])}')
    print('#' * 70)

    # 1. Write sidecar params (read by the injected cell)
    PARAMS_FILE.write_text(json.dumps({
        'scenario': name,
        'prices': cfg_s['prices'],
        'bound_patches': cfg_s.get('bound_patches', []),
    }, indent=2), encoding='utf-8')

    # 2. Build patched optimizer notebook
    nb_path = prepare_optimizer_nb(scenario_dir)
    print(f'  patched optimizer: {nb_path.relative_to(ROOT)}')

    # 3. Execute optimizer
    t0 = time.time()
    try:
        nb = nbformat.read(str(nb_path), as_version=4)
        ep = ExecutePreprocessor(timeout=EXECUTE_TIMEOUT, kernel_name='python3')
        ep.preprocess(nb, {'metadata': {'path': str(ROOT)}})
        nbformat.write(nb, str(nb_path))
        opt_seconds = round(time.time() - t0, 1)
        opt_status = 'ok'
        print(f'  optimizer OK ({opt_seconds}s)')
    except Exception as e:
        opt_seconds = round(time.time() - t0, 1)
        opt_status = 'failed'
        print(f'  optimizer FAILED ({opt_seconds}s): {e}')
        # still try post-process so we capture whatever was written
        try: nbformat.write(nb, str(nb_path))
        except Exception: pass

    # 4. Run post-processor (independent script)
    post_status = 'skipped'
    post_seconds = 0.0
    if not SKIP_POST_PROCESS:
        t1 = time.time()
        try:
            r = subprocess.run([sys.executable, str(POST_SCRIPT)],
                               cwd=str(ROOT), capture_output=True, text=True,
                               timeout=900)
            post_seconds = round(time.time() - t1, 1)
            if r.returncode == 0:
                post_status = 'ok'
                print(f'  post_process OK ({post_seconds}s)')
            else:
                post_status = 'failed'
                print(f'  post_process FAILED rc={r.returncode} ({post_seconds}s)')
                print('  stderr:', (r.stderr or '')[-800:])
        except subprocess.TimeoutExpired:
            post_status = 'timeout'
            post_seconds = round(time.time() - t1, 1)
            print(f'  post_process TIMEOUT after {post_seconds}s')
        except Exception as e:
            post_status = 'error'
            print(f'  post_process ERROR: {e}')

    # 5. Snapshot artifacts
    captured = _snapshot_outputs(scenario_dir, started_at=started)
    parsed   = _parse_model_output(scenario_dir)

    # 6. Tidy: remove the params file (so a manual re-run of the optimizer is canonical)
    try: PARAMS_FILE.unlink()
    except FileNotFoundError: pass

    return {
        'scenario': name,
        'description': cfg_s.get('description',''),
        'power_rate': cfg_s['prices'].get('Power_Rate'),
        'fuel_rate':  cfg_s['prices'].get('Fuel_Rate'),
        'dmw_rate':   cfg_s['prices'].get('DMW_Rate'),
        'n_bound_patches': len(cfg_s.get('bound_patches', [])),
        'optimizer_status': opt_status, 'optimizer_seconds': opt_seconds,
        'post_status': post_status, 'post_seconds': post_seconds,
        'captured': captured,
        **parsed,
        'scenario_dir': str(scenario_dir),
    }

print('Runner ready.')


In [ ]:
# ── Run all scenarios ────────────────────────────────────────────────────────
results, errors = [], []
t_run0 = time.time()

for SCENARIO in SCENARIOS_TO_RUN:
    try:
        results.append(run_scenario(SCENARIO))
    except Exception as e:
        tb = traceback.format_exc()
        errors.append({'scenario': SCENARIO, 'error': str(e), 'traceback': tb})
        print(f'\n>>> {SCENARIO}: DRIVER ERROR — {e}')
        if not CONTINUE_ON_ERROR: raise
        results.append({'scenario': SCENARIO, 'optimizer_status': 'driver_error',
                        'description': SCENARIO_CONFIGS[SCENARIO].get('description',''),
                        'error': str(e)})

print(f'\n\nTotal wall time: {(time.time()-t_run0)/60:.1f} min  |  '
      f'optimizer OK: {sum(1 for r in results if r.get("optimizer_status")=="ok")}/{len(SCENARIOS_TO_RUN)}')


In [ ]:
# ── Combined cross-scenario report ──────────────────────────────────────────
import xlsxwriter

_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
combined_path = ROOT / f'eo_all_scenarios_{_ts}.xlsx'
wb = xlsxwriter.Workbook(str(combined_path))

def _f(d): return wb.add_format(d)
F = {
    'title':  _f({'bold': True, 'font_size': 16, 'font_color': '#1F3864'}),
    'header': _f({'bold': True, 'bg_color': '#D6E4F0', 'border': 1, 'align': 'center'}),
    'label':  _f({'bold': True, 'bg_color': '#F2F2F2', 'border': 1}),
    'text':   _f({'border': 1}),
    'value':  _f({'border': 1, 'num_format': '#,##0.00'}),
    'value4': _f({'border': 1, 'num_format': '#,##0.0000'}),
    'pct':    _f({'border': 1, 'num_format': '0.00%'}),
    'int':    _f({'border': 1, 'num_format': '0'}),
    'green':  _f({'bold': True, 'bg_color': '#E2EFDA', 'font_color': '#375623',
                  'border': 1, 'num_format': '#,##0.00'}),
    'red':    _f({'bold': True, 'bg_color': '#FCE4D6', 'font_color': '#9C0006',
                  'border': 1, 'num_format': '#,##0.00'}),
    'note':   _f({'italic': True, 'font_color': '#595959', 'font_size': 9}),
}

ws = wb.add_worksheet('Summary')
ws.set_column(0, 0, 20); ws.set_column(1, 1, 28); ws.set_column(2, 14, 14)
ws.write(0, 0, 'EO All-Scenarios Driver — Summary', F['title'])
ws.write(1, 0, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}', F['note'])
ws.write(2, 0, f'Scenarios run: {len(results)}  |  Optimizer OK: '
                f'{sum(1 for r in results if r.get("optimizer_status")=="ok")}', F['note'])

cols = ['Scenario', 'Description', 'Opt Status', 'Post Status',
        'Power', 'Fuel', 'DMW',
        'Baseline $/hr', 'Optimum $/hr', 'Saving $/hr', 'Saving %', 'Annual $M/yr',
        'Opt sec', 'Post sec']
for c, h in enumerate(cols): ws.write(4, c, h, F['header'])

r = 5
for res in results:
    ws.write(r, 0, res.get('scenario',''), F['label'])
    ws.write(r, 1, res.get('description',''), F['text'])
    opt_s = str(res.get('optimizer_status','?'))
    ws.write(r, 2, opt_s, F['green'] if opt_s=='ok' else F['red'])
    ws.write(r, 3, str(res.get('post_status','?')), F['text'])
    def _wn(col, key, fmt):
        v = res.get(key)
        if v is None or (isinstance(v,float) and v != v):
            ws.write(r, col, '', F['text'])
        else:
            ws.write(r, col, float(v), fmt)
    _wn(4,'power_rate', F['value4']); _wn(5,'fuel_rate', F['value4']); _wn(6,'dmw_rate', F['value4'])
    _wn(7,'baseline_obj', F['value']); _wn(8,'optimum_obj', F['value'])
    sv = res.get('saving')
    if sv is None or (isinstance(sv,float) and sv != sv):
        ws.write(r, 9, '', F['text'])
    else:
        ws.write(r, 9, float(sv), F['green'] if float(sv) >= 0 else F['red'])
    spct = res.get('saving_pct')
    if spct is None or (isinstance(spct,float) and spct != spct):
        ws.write(r,10,'',F['text'])
    else:
        ws.write(r,10, float(spct)/100.0, F['pct'])
    ann = (float(sv) * 8760 / 1e6) if (sv is not None and not (isinstance(sv,float) and sv!=sv)) else None
    if ann is None: ws.write(r,11,'',F['text'])
    else: ws.write(r,11,ann,F['value4'])
    _wn(12,'optimizer_seconds', F['value'])
    _wn(13,'post_seconds', F['value'])
    r += 1

ws.write(r+1, 0, 'Per-scenario artifacts in scenario_runs/<name>/', F['note'])

if errors:
    we = wb.add_worksheet('Errors')
    we.set_column(0,0,20); we.set_column(1,1,80)
    we.write(0,0,'Scenario Errors',F['title'])
    we.write(2,0,'Scenario',F['header']); we.write(2,1,'Error',F['header'])
    for ri,e in enumerate(errors, start=3):
        we.write(ri,0,e['scenario'],F['label']); we.write(ri,1,e['error'],F['text'])

wb.close()
print(f'\nCombined report: {combined_path.name}')
print(f'Per-scenario artifacts: {RUNS_DIR}/')


In [ ]:
# ── Final summary ───────────────────────────────────────────────────────────
print('=' * 72)
print('  ALL-SCENARIOS DRIVER — DONE')
print('=' * 72)
for res in results:
    s   = str(res.get('optimizer_status','?')).upper()
    sv  = res.get('saving')
    sv_s = f'${float(sv):,.2f}/hr' if sv is not None and not (isinstance(sv,float) and sv!=sv) else 'n/a'
    print(f'  {res["scenario"]:<20} OPT={s:<6} POST={str(res.get("post_status","?")).upper():<6} saving={sv_s}')
print('=' * 72)
print(f'  Combined: {combined_path.name}')
print(f'  Folder:   {RUNS_DIR}/')
print('=' * 72)
